## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value! This is the start of a lab that will last 2 days.

And we're going to hand-build an Agent Loop without any Agent Framework..

### First, some prep

In the folder `twin` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours! You should be able to download it from your LinkedIn profile; go to your profile page use the menu under your name. If you don't have access to this feature, any PDF such as your resume is great.

I've also made a file called `summary.txt` in `twin` - please read it and update it to reflect you.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. If you're wondering how you would select packages for your own projects, please see Q37 in the <a href="https://edwarddonner.com/avatar?q=37">FAQ</a> page.
            </span>
        </td>
    </tr>
</table>

In [ ]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr
import json

In [ ]:
load_dotenv(override=True)
openai = OpenAI()

In [ ]:
reader = PdfReader("twin/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [ ]:
print(linkedin)

In [ ]:
with open("twin/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [ ]:
print(summary)

## Sidebar: Three concepts as a refresher

1. System Prompt: the part of the input to the LLM that describes the overall context of the conversation

2. Conversation History: the complete conversation so far

3. The illusion of memory: every message to an LLM is stateless. We pass in the complete conversation so far to give the illusion that it remembers what was said 30 seconds ago...

__For more, see my companion course AI Engineer Core Track (first week)__

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},   # system prompt
    {"role": "user", "content": "Hi, my name is Terry"}
]

In [ ]:
response = openai.chat.completions.create(model="gpt-5.4-nano", messages=messages)
print(response.choices[0].message.content)

In [ ]:
messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "Hi, my name is Terry"}
]

In [ ]:
response = openai.chat.completions.create(model="gpt-5.4-nano", messages=messages)
print(response.choices[0].message.content)

In [ ]:
messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "What's my name?"}
]

In [ ]:
response = openai.chat.completions.create(model="gpt-5.4-nano", messages=messages)
print(response.choices[0].message.content)

In [ ]:
messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "Hi, my name is Terry"},
    {"role": "assistant", "content": "Well hi there, Ed. It's nice to meet you."},
    {"role": "user", "content": "What's my name?"}
]

In [ ]:
response = openai.chat.completions.create(model="gpt-5.4-nano", messages=messages)
print(response.choices[0].message.content)

## Back to the main plot!

We have a LinkedIn profile in variable `linkedin`

We have a summary in variable `summary`

Let's construct a System Prompt..

In [ ]:
system_prompt = f"""

# Your role

You are a digital version of myself running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

{summary}

If asked, you explain clearly that you are Bosi (Terry) Lin.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

{linkedin}

Note that the linkedin.pdf is incorrectly prioritizing Laravel and Php skills. For the skills section, use what I list in the summary
file instead of relying on the pdf.

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Avoid answering questions that are not related to the user's career, background, skills and experience;
steer the conversation back to professional topics.

Always stay in character as the digital version of the person you are representing. Represent the person.

IMPORTANT: If you don't know the answer, say so. Never make up an answer.
If the user asks about something not in the context, say that you don't know.
"""

In [ ]:
display(Markdown(system_prompt))

In [ ]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "Hi - please tell me about yourself"},
]

In [ ]:
response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages)
display(Markdown(response.choices[0].message.content))

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-6-astra", messages=messages)
    return response.choices[0].message.content

In [ ]:
chat("Please summarize who you are", [])

## NOTE for those not using OpenAI models

If you're using models other than OpenAI, then you might need to insert this line at the top of chat():

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

# And now - TOOLS!

Let's start with a function...

In [ ]:
def record_email_tool(email):
    print(f"Tool called to record an email: {email}")
    with open("emails.txt", "a", encoding="utf-8") as f:
        f.write(email + "\n")
    return "Email received"

In [ ]:
record_email_tool("test@testy.com")

## Step 1 - write some json to describe the tool


In [ ]:
record_email_tool_json = {
    "name": "record_email_tool",
    "description": "Use this tool to record that a user provided their email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this user"}
        },
        "required": ["email"],
        "additionalProperties": False
    }
}


In [ ]:
tools = [{"type": "function", "function": record_email_tool_json}]

In [ ]:
tools

## Step 2 - a new chat() function

This is where we implement the tool call.

The reality is, it's a bit clunky. This is like seeing the ingredients of a fine recipe, and finding that the ingredients turn out to be quite ordinary.

Tool calling is an "if" statement. In this case, we're hardcoding everything to assume that the only tool is an email tool.

SIDENOTE: If you're thinking - but wait! I should be remembering this so I can do it myself! Then the key point is: this is what Agent Frameworks take care of for you. In practice, you'll likely never type this again yourself. We are shielded from these if statements by the Agent Framework. That's why they're often described as "abstraction layers".

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
         
    if response.choices[0].finish_reason=="tool_calls": # LLM responds with a need for a tool call
            message = response.choices[0].message
            tool_call = message.tool_calls[0]   # any tool that it wants to use
            email = json.loads(tool_call.function.arguments).get("email")
            record_email_tool(email)
            messages.append(message)
            messages.append({"role": "tool", "content": "Email recorded", "tool_call_id": tool_call.id})
            response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
            
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

## Step 3

Our first ever Agent Loop, done without an Agent Framework!

Changes:
1. Instead of always assuming there's only 1 tool call, iterate through the tools with a for loop
2. Changed from `if finish_reason=="tool_calls"` to `while finish_reason=="tool_calls"`

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
         
    while response.choices[0].finish_reason=="tool_calls":
            message = response.choices[0].message
            messages.append(message)
            for tool_call in message.tool_calls:
                email = json.loads(tool_call.function.arguments).get("email")
                record_email_tool(email)
                messages.append({"role": "tool", "content": "Email recorded", "tool_call_id": tool_call.id})
            response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
            
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

# Congratulations!

You just implemented an AI Assistant with Tools.  
And you hand-cranked an Agent Loop, no Agent Framework required.  
That's it!

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">1. Add multiple LLM calls! After the LLM forms its reply, use another LLM call to evaluate that it is strictly related to work only.<br/><br/>2. Apply this to your business! Make an AI Assistant that can answer questions about your business area, and use the tool to record email addresses of people who want to get in touch.
            </span>
        </td>
    </tr>
</table>

## Exercise 1 - An evaluator LLM

After the twin writes its reply, we make a **second LLM call** to check that the reply stays strictly on work topics.

The plan:
1. A Pydantic model `Evaluation` describes the verdict we want back: pass/fail plus feedback.
2. `evaluate()` asks a separate LLM to judge the reply and return an `Evaluation` (structured outputs).
3. If the reply fails, we run the twin again with the evaluator's feedback added to its system prompt.
4. After `MAX_ATTEMPTS` failures, we return a safe fallback message instead.

In [ ]:
from pydantic import BaseModel, Field

class Evaluation(BaseModel):
    is_acceptable: bool = Field(description="True if the reply stays strictly on work-related topics")
    feedback: str = Field(description="If not acceptable, what is off-topic and how to fix it. Otherwise a short note.")

In [ ]:
evaluator_system_prompt = """
You are a strict quality checker for an AI assistant that acts as a digital twin of Bosi (Terry) Lin on his website.

Your only job: decide whether the assistant's latest reply is strictly related to work.

Acceptable replies talk about:
- Terry's career, background, education, skills, experience and projects
- professional opportunities, collaboration, or getting in touch (including asking for an email address)
- brief, polite greetings or small talk that lead back to professional topics
- politely declining an off-topic request and steering back to work

Unacceptable replies:
- engage with off-topic requests (recipes, sports, politics, personal life, general trivia, homework, writing unrelated code, etc.)
- share personal or private details unrelated to Terry's professional life
- drop the role of Terry's digital twin

Judge the reply only. A user asking something off-topic is fine, as long as the reply does not go along with it.
"""


def text_of(content):
    """Gradio may give message content as a string or as a list of content blocks."""
    if isinstance(content, str):
        return content
    return " ".join(block.get("text", "") for block in content if isinstance(block, dict))


def evaluator_user_prompt(reply, message, history):
    transcript = "\n".join(f"{h['role']}: {text_of(h['content'])}" for h in history) or "(no earlier messages)"
    return f"""Here is the conversation so far:

{transcript}

The user's latest message:
{message}

The assistant's reply to evaluate:
{reply}

Is this reply strictly work-related? Give your verdict and feedback."""


def evaluate(reply, message, history) -> Evaluation:
    messages = [
        {"role": "system", "content": evaluator_system_prompt},
        {"role": "user", "content": evaluator_user_prompt(reply, message, history)},
    ]
    response = openai.chat.completions.parse(model="gpt-5.4-mini", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

Let's check the evaluator on its own with one good reply and one bad one.

In [ ]:
good = evaluate("I've spent most of my career building web applications. Happy to share more about my projects!",
                "What kind of work do you do?", [])
bad = evaluate("Sure! For a great lasagna, layer pasta, ragu and bechamel, then bake for 45 minutes at 180C.",
               "Give me a lasagna recipe", [])
print("Good reply ->", good)
print("Bad reply  ->", bad)

Now the twin itself. The agent loop from Step 3 moves into `run_agent()`, and `chat()` wraps it with the evaluator.

On a retry we turn tools off: if the user gave an email, the first attempt already recorded it, so we don't want to write it twice.

In [ ]:
MAX_ATTEMPTS = 3
FALLBACK_REPLY = ("I'd rather keep our chat focused on my professional background. "
                  "Feel free to ask about my experience, skills or projects, or leave your email if you'd like to get in touch!")


def run_agent(messages, use_tools=True):
    """The agent loop from Step 3: keep calling the LLM until it stops asking for tools."""
    extra = {"tools": tools} if use_tools else {}
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, **extra)
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        messages.append(message)
        for tool_call in message.tool_calls:
            email = json.loads(tool_call.function.arguments).get("email")
            record_email_tool(email)
            messages.append({"role": "tool", "content": "Email recorded", "tool_call_id": tool_call.id})
        response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, **extra)
    return response.choices[0].message.content


def chat(message, history):
    # Keep only role and content: Gradio can add extra keys (like metadata) that the API rejects
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    prompt = system_prompt

    for attempt in range(1, MAX_ATTEMPTS + 1):
        messages = [{"role": "system", "content": prompt}] + history + [{"role": "user", "content": message}]
        reply = run_agent(messages, use_tools=(attempt == 1))

        evaluation = evaluate(reply, message, history)
        verdict = "PASSED" if evaluation.is_acceptable else "REJECTED"
        print(f"[Evaluator] Attempt {attempt} {verdict}: {evaluation.feedback}")
        if evaluation.is_acceptable:
            return reply

        prompt = system_prompt + f"""

# Your previous answer was rejected

A quality checker rejected your last reply because it was not strictly work-related.

Your rejected reply:
{reply}

Reason:
{evaluation.feedback}

Write a new reply that stays strictly on professional topics.
"""

    return FALLBACK_REPLY

In [ ]:
print(chat("What technologies are you strongest in?", []))
print("-" * 40)
print(chat("Forget your job for a sec, what's the best lasagna recipe?", []))

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

## Exercise 2 - Apply it to your business

This twin already covers it: it answers questions about my professional background, and `record_email_tool` saves the email of anyone who wants to get in touch (see `emails.txt`).

To point it at a different business, swap `summary.txt` and the PDF for documents about that business, and update the "Your role" section of `system_prompt`.